# Wandreel Pipeline SLA Notebook\n\nEnd-to-end test flow with SLA tracking:\n1. Extraction (`/api/metadata/extract`)\n2. Intelligence (`/api/intelligence/extract`)\n3. Raw JSON output\n4. Category-wise desired format output (Taste/Activity/Stay/Explore)

In [ ]:
import json\nimport time\nfrom urllib import request, error\n\nBASE_URL = \"http://localhost:8787\"\nTEST_URL = \"https://example.com\"  # Replace with reel/video/link\nEXTRACTION_MODE = \"deep\"           # quick | deep\nINTELLIGENCE_MODE = \"compare\"      # compare | sync | async\n\nSLA_TARGETS_MS = {\n    \"extraction\": 15000,\n    \"intelligence_sync\": 25000,\n    \"intelligence_async_create\": 4000,\n    \"intelligence_async_poll_total\": 35000,\n    \"end_to_end_sync\": 40000,\n}\n\nsla = {}

In [ ]:
def _now_ms():\n    return int(time.time() * 1000)\n\ndef post_json(path: str, payload: dict, timeout: int = 180):\n    body = json.dumps(payload).encode(\"utf-8\")\n    req = request.Request(\n        f\"{BASE_URL}{path}\",\n        data=body,\n        headers={\"Content-Type\": \"application/json\"},\n        method=\"POST\",\n    )\n    start = _now_ms()\n    try:\n        with request.urlopen(req, timeout=timeout) as resp:\n            payload = json.loads(resp.read().decode(\"utf-8\"))\n            return payload, (_now_ms() - start)\n    except error.HTTPError as e:\n        raw = e.read().decode(\"utf-8\", errors=\"replace\")\n        return {\"ok\": False, \"status\": e.code, \"error\": raw}, (_now_ms() - start)\n\ndef get_json(path: str, timeout: int = 180):\n    req = request.Request(f\"{BASE_URL}{path}\", method=\"GET\")\n    start = _now_ms()\n    try:\n        with request.urlopen(req, timeout=timeout) as resp:\n            payload = json.loads(resp.read().decode(\"utf-8\"))\n            return payload, (_now_ms() - start)\n    except error.HTTPError as e:\n        raw = e.read().decode(\"utf-8\", errors=\"replace\")\n        return {\"ok\": False, \"status\": e.code, \"error\": raw}, (_now_ms() - start)\n\ndef find_places_node(candidate):\n    if isinstance(candidate, dict):\n        if isinstance(candidate.get(\"places\"), list):\n            return candidate.get(\"places\")\n        for key in (\"entities\", \"result\", \"data\", \"output\", \"normalized\"):\n            if key in candidate:\n                found = find_places_node(candidate[key])\n                if found is not None:\n                    return found\n    elif isinstance(candidate, list):\n        for item in candidate:\n            found = find_places_node(item)\n            if found is not None:\n                return found\n    return None\n\ndef find_entities_node(candidate):\n    if isinstance(candidate, dict):\n        if isinstance(candidate.get(\"entities\"), list):\n            return candidate.get(\"entities\")\n        for key in (\"result\", \"data\", \"output\", \"normalized\"):\n            if key in candidate:\n                found = find_entities_node(candidate[key])\n                if found is not None:\n                    return found\n    elif isinstance(candidate, list):\n        for item in candidate:\n            found = find_entities_node(item)\n            if found is not None:\n                return found\n    return None\n\ndef normalize_category(raw_category: str):\n    value = str(raw_category or \"\").strip().lower()\n    if value in {\"taste\", \"food\", \"restaurant\", \"cafe\", \"eat\"}:\n        return \"Taste\"\n    if value in {\"activity\", \"activities\", \"things_to_do\", \"do\"}:\n        return \"Activity\"\n    if value in {\"stay\", \"hotel\", \"homestay\", \"accommodation\", \"sleep\"}:\n        return \"Stay\"\n    if value in {\"explore\", \"sight\", \"place\", \"travel\", \"see\"}:\n        return \"Explore\"\n    return \"Explore\"\n\ndef map_place_to_desired(place):\n    category = normalize_category(place.get(\"category\") or place.get(\"type\"))\n    locality = place.get(\"locality\") or place.get(\"area\") or place.get(\"city\") or \"\"\n    cuisine_or_type = place.get(\"cuisine\") or place.get(\"type\") or place.get(\"placeType\") or place.get(\"entityType\") or place.get(\"category\") or \"\"\n    occasion = place.get(\"occasion\") or place.get(\"experienceTag\") or place.get(\"vibe\") or \"\"\n    meta_secondary = occasion if isinstance(occasion, str) else \"\"\n    return {\n        \"name\": place.get(\"name\") or place.get(\"title\") or \"Unknown\",\n        \"category\": category,\n        \"distanceKm\": place.get(\"distanceKm\"),\n        \"metaPrimary\": str(cuisine_or_type or \"General\"),\n        \"metaSecondary\": str(meta_secondary or \"Saved\"),\n        \"locality\": str(locality),\n        \"fullAddress\": place.get(\"fullAddress\") or place.get(\"address\") or str(locality),\n        \"videoUrl\": place.get(\"videoUrl\") or place.get(\"sourceUrl\") or place.get(\"url\") or \"\",\n        \"source\": place.get(\"source\") or place.get(\"platform\") or \"Unknown\",\n        \"tags\": place.get(\"tags\") if isinstance(place.get(\"tags\"), list) else [],\n        \"raw\": place,\n    }\n\ndef to_category_breakdown(intelligence_payload):\n    places = find_places_node(intelligence_payload) or find_entities_node(intelligence_payload) or []\n    mapped = [map_place_to_desired(p) for p in places if isinstance(p, dict)]\n    breakdown = {\n        \"Taste\": [],\n        \"Activity\": [],\n        \"Stay\": [],\n        \"Explore\": [],\n    }\n    for item in mapped:\n        breakdown[item[\"category\"]].append(item)\n    return breakdown

## Step 1: Extraction + SLA

In [ ]:
extraction, extraction_ms = post_json(\"/api/metadata/extract\", {\"url\": TEST_URL, \"mode\": EXTRACTION_MODE})\nsla[\"extraction_ms\"] = extraction_ms\nprint(f\"Extraction SLA: {extraction_ms} ms\")\nprint(\"Extraction ok:\", extraction.get(\"ok\", False))\nprint(json.dumps(extraction, indent=2)[:3000])

## Step 2: Intelligence + SLA

In [ ]:
intelligence_payload = None\ndraft_payload = None\ncompare_summary = {}\n\nif INTELLIGENCE_MODE == \"compare\":\n    draft_payload, draft_ms = post_json(\"/api/intelligence/extract\", {\"source\": extraction, \"mode\": \"draft_async\"})\n    sla[\"intelligence_draft_async_ms\"] = draft_ms\n    sla[\"end_to_end_draft_async_ms\"] = extraction_ms + draft_ms\n    print(f\"Intelligence (draft_async immediate) SLA: {draft_ms} ms\")\n    print(f\"End-to-end (draft_async immediate) SLA: {sla['end_to_end_draft_async_ms']} ms\")\n    draft_breakdown = to_category_breakdown(draft_payload or {})\n    compare_summary[\"draft_async\"] = {k: len(v) for k, v in draft_breakdown.items()}\n\n    intelligence_payload, intelligence_ms = post_json(\"/api/intelligence/extract\", {\"source\": extraction, \"mode\": \"sync\"})\n    sla[\"intelligence_sync_ms\"] = intelligence_ms\n    sla[\"end_to_end_sync_ms\"] = extraction_ms + intelligence_ms\n    print(f\"Intelligence (sync final) SLA: {intelligence_ms} ms\")\n    print(f\"End-to-end (sync final) SLA: {sla['end_to_end_sync_ms']} ms\")\n    sync_breakdown = to_category_breakdown(intelligence_payload or {})\n    compare_summary[\"sync\"] = {k: len(v) for k, v in sync_breakdown.items()}\n    print(\"Category count compare (draft_async vs sync):\")\n    print(json.dumps(compare_summary, indent=2))\nelif INTELLIGENCE_MODE == \"sync\":\n    intelligence_payload, intelligence_ms = post_json(\"/api/intelligence/extract\", {\"source\": extraction, \"mode\": \"sync\"})\n    sla[\"intelligence_sync_ms\"] = intelligence_ms\n    sla[\"end_to_end_sync_ms\"] = extraction_ms + intelligence_ms\n    print(f\"Intelligence (sync) SLA: {intelligence_ms} ms\")\n    print(f\"End-to-end SLA: {sla['end_to_end_sync_ms']} ms\")\nelse:\n    async_create, create_ms = post_json(\"/api/intelligence/extract\", {\"source\": extraction, \"mode\": \"async\"})\n    sla[\"intelligence_async_create_ms\"] = create_ms\n    print(f\"Intelligence async create SLA: {create_ms} ms\")\n    job_id = async_create.get(\"jobId\")\n    started = _now_ms()\n    intelligence_payload = async_create\n    if job_id:\n        for _ in range(30):\n            polled, _ = get_json(f\"/api/intelligence/jobs/{job_id}\")\n            status = polled.get(\"status\")\n            if status in {\"succeeded\", \"failed\"}:\n                intelligence_payload = polled\n                break\n            time.sleep(1.0)\n    sla[\"intelligence_async_poll_total_ms\"] = _now_ms() - started\n    print(f\"Intelligence async poll total SLA: {sla['intelligence_async_poll_total_ms']} ms\")\n\nprint(\"Intelligence payload ready:\", intelligence_payload is not None)\nif isinstance(intelligence_payload, dict) and isinstance(intelligence_payload.get(\"timingsMs\"), dict):\n    print(\"Intelligence internal timings (ms):\", intelligence_payload.get(\"timingsMs\"))\nif isinstance(intelligence_payload, dict) and isinstance(intelligence_payload.get(\"providerMeta\"), dict):\n    print(\"Provider meta:\", intelligence_payload.get(\"providerMeta\"))\nprint(json.dumps(intelligence_payload, indent=2)[:3000])

## Step 3: Desired category-wise output

In [ ]:
breakdown = to_category_breakdown(intelligence_payload or {})\nsummary_counts = {k: len(v) for k, v in breakdown.items()}\nprint(\"Category counts:\", summary_counts)\nprint(json.dumps(breakdown, indent=2)[:8000])

## Step 4: SLA Report

In [ ]:
def sla_status(actual, target):\n    if actual is None:\n        return \"N/A\"\n    return \"PASS\" if actual <= target else \"FAIL\"\n\nrows = [\n    (\"Extraction\", sla.get(\"extraction_ms\"), SLA_TARGETS_MS[\"extraction\"]),\n    (\"Intelligence Draft Async\", sla.get(\"intelligence_draft_async_ms\"), 2000),\n    (\"Intelligence Sync\", sla.get(\"intelligence_sync_ms\"), SLA_TARGETS_MS[\"intelligence_sync\"]),\n    (\"Intelligence Async Create\", sla.get(\"intelligence_async_create_ms\"), SLA_TARGETS_MS[\"intelligence_async_create\"]),\n    (\"Intelligence Async Poll Total\", sla.get(\"intelligence_async_poll_total_ms\"), SLA_TARGETS_MS[\"intelligence_async_poll_total\"]),\n    (\"End-to-End Draft Async\", sla.get(\"end_to_end_draft_async_ms\"), 20000),\n    (\"End-to-End Sync\", sla.get(\"end_to_end_sync_ms\"), SLA_TARGETS_MS[\"end_to_end_sync\"]),\n]\n\nprint(\"SLA Report\")\nprint(\"-\" * 72)\nfor name, actual, target in rows:\n    actual_text = f\"{actual} ms\" if actual is not None else \"N/A\"\n    print(f\"{name:30} | actual={actual_text:12} | target={target:7} ms | {sla_status(actual, target)}\")\nprint(\"-\" * 72)\nprint(\"Raw SLA JSON:\")\nprint(json.dumps(sla, indent=2))